# Shipping Route Efficiency Analysis for Nassau Candy Distributor

##### **Project** :  Nassau candy distributer shipping route efficiency analysis
##### **Author**  :  Shreyash Ambade
##### **Date**    :  28-03-2026

In [1]:
# importng required libraries for numerical operations ,data manipulations and data visualization

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
from datetime import datetime

## 1. Data Cleaning and Validation

### loading dataset and seeing its features

In [2]:
# loading the CSV file into pandas dataframe(df)

df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\Nassau Candy Distributor.csv")

# look at the data
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [3]:
# see how many rows and columns we have
df.shape

(10194, 18)

In [4]:
# see information and data types of each column
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

In [5]:
# count the null values in each column 
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64

In [6]:
# checking numbers columns are valid
print((df['Sales'] < 0).sum())
print((df['Gross Profit'] < 0).sum())
print((df['Cost'] < 0).sum())

0
0
0


### • Validate date formats

In [8]:
# validate date formats (converting string data format to real datetime format)

df['Order Date'] = pd.to_datetime(df['Order Date'],format = '%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'],format = '%d-%m-%Y')

In [9]:
#  confirming the dtypes we change before
df.dtypes

Row ID                     int64
Order ID                  object
Order Date        datetime64[ns]
Ship Date         datetime64[ns]
Ship Mode                 object
Customer ID                int64
Country/Region            object
City                      object
State/Province            object
Postal Code               object
Division                  object
Region                    object
Product ID                object
Product Name              object
Sales                    float64
Units                      int64
Gross Profit             float64
Cost                     float64
dtype: object

### • Remove invalid or negative lead times

In [10]:
# calculate shipping lead time 
df['Lead Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# remove invalid or negative lead times
df = df[df['Lead Time'] >= 0]

In [11]:
# seeing result

df['Lead Time'].head()

0    909
1    909
2    909
3    909
4    912
Name: Lead Time, dtype: int64

### • Handle missing shipment records

In [12]:
# seeing missing shipment records
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
Lead Time         0
dtype: int64

In [13]:
# here is no any missing values but for our concerns
df = df.dropna()

In [14]:
# confirming that there is no missing value
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
Lead Time         0
dtype: int64

In [15]:
# checking basic stats 
df['Lead Time'].describe()

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Lead Time, dtype: float64

### • Standardize geographic fields

In [16]:
# Standardising Geographic fields (cleaning text columns).(removing extra space and fix capitalization)

# removing extra space
df['Country/Region'] = df['Country/Region'].str.strip()
df['City'] = df['City'].str.strip()
df['State/Province'] = df['State/Province'].str.strip()
df['Region'] = df['Region'].str.strip()
df['Product Name'] = df['Product Name'].str.strip()
df['Ship Mode'] = df['Ship Mode'].str.strip()

# converting into consistent format
df['Country/Region'] = df['Country/Region'].str.title()
df['City'] = df['City'].str.title()
df['State/Province'] = df['State/Province'].str.title()
df['Region'] = df['Region'].str.title()
df['Product Name'] = df['Product Name'].str.title()
df['Ship Mode'] = df['Ship Mode'].str.title()


In [17]:
# checking info of dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  object        
 2   Order Date      10194 non-null  datetime64[ns]
 3   Ship Date       10194 non-null  datetime64[ns]
 4   Ship Mode       10194 non-null  object        
 5   Customer ID     10194 non-null  int64         
 6   Country/Region  10194 non-null  object        
 7   City            10194 non-null  object        
 8   State/Province  10194 non-null  object        
 9   Postal Code     10194 non-null  object        
 10  Division        10194 non-null  object        
 11  Region          10194 non-null  object        
 12  Product ID      10194 non-null  object        
 13  Product Name    10194 non-null  object        
 14  Sales           10194 non-null  float64       
 15  Un

In [18]:
#  validating   the steps i ferformed before
print("total records :",len(df))
print("negative lead time :",(df['Lead Time'] < 0).sum())
print("missing values:\n",df.isnull().sum())

total records : 10194
negative lead time : 0
missing values:
 Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
Lead Time         0
dtype: int64


In [ ]:
# # save clean data to new  csv file
# df.to_csv('nassau_data_clean_&_validation.csv',index=False)

# 2. Feature Engineering

### Loading Dataset

In [20]:
# loading  dataset in pandas DataFrame and read it

df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_data_clean_&_validation.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912


In [19]:
# see all columns

df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time'],
      dtype='object')

In [21]:
# checing info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

### Categorizing Routes

In [22]:
# first we need to creat new feature called factory from given details (mapping each products to its factory)

factoy_mapping = {
    "Wonka Bar - Nutty Crunch Surprise":"Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows":"Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious":"Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate":"Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel":"Wicked Choccy's",
    "Laffy Taffy":"Sugar Shack",
    "SweeTARTS":"Sugar Shack",
    "Nerds":"Sugar Shack",
    "Fun Dip":"Sugar Shack",
    "Fizzy Lifting Drinks":"Sugar Shack",
    "Everlasting Gobstopper":"Secret Factory",
    "Hair Toffee":"The Other Factory",
    "Lickable Wallpaper":"Secret Factory",
    "Wonka Gum":"Secret Factory",
    "Kazookles":"The Other Factory"
}

# creating new feture df['Factory']

df['Factory'] = df['Product Name'].map(factoy_mapping)

In [23]:
# checking if any product didnt get mapped

df['Factory'].isnull().sum()

np.int64(10)

In [24]:
# finding unmapped products
# rows wher factory is empyt
missing = df[df['Factory'].isnull()]
# see which product are cousing the problem
missing['Product Name'].unique()

array(['Sweetarts'], dtype=object)

In [25]:
# mapping again aby adding product name 'Sweetarts'
factoy_mapping = {
    "Wonka Bar - Nutty Crunch Surprise":"Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows":"Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious":"Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate":"Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel":"Wicked Choccy's",
    "Laffy Taffy":"Sugar Shack",
    "SweeTARTS":"Sugar Shack",
    "Sweetarts":"Sugar Shack",
    "Nerds":"Sugar Shack",
    "Fun Dip":"Sugar Shack",
    "Fizzy Lifting Drinks":"Sugar Shack",
    "Everlasting Gobstopper":"Secret Factory",
    "Hair Toffee":"The Other Factory",
    "Lickable Wallpaper":"Secret Factory",
    "Wonka Gum":"Secret Factory",
    "Kazookles":"The Other Factory"
}

# creating new feture df['Factory']

df['Factory'] = df['Product Name'].map(factoy_mapping)

In [26]:
# again checking if any product didnt get mapped

df['Factory'].isnull().sum()

np.int64(0)

In [27]:
# checking the feature was a created  or not(checking the result)
df[['Factory','Product Name']].head()

,Factory,Product Name
0,Wicked Choccy's,Wonka Bar - Milk Chocolate
1,Wicked Choccy's,Wonka Bar - Triple Dazzle Caramel
2,Lot's O' Nuts,Wonka Bar - Nutty Crunch Surprise
3,Lot's O' Nuts,Wonka Bar -Scrumdiddlyumptious
4,Wicked Choccy's,Wonka Bar - Triple Dazzle Caramel


In [28]:
#  count missing or NaN values of each column
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
Lead Time         0
Factory           0
dtype: int64

###   Categorizing routes by:
#####   ○ Factory → Customer Region
#####   ○ Factory → Customer State

In [29]:
# Categorization of Routes(factory -> customers)

# 1. factory --> customer region
df['Route_Region'] = df['Factory'] + "-->" + df['Region']
                      
# 2. factory --> customer state
df['Route_State'] = df['Factory'] + "-->" + df['State/Province']

In [30]:
# see the result 
df[['Factory','Region','State/Province','Route_Region','Route_State']].head()

,Factory,Region,State/Province,Route_Region,Route_State
0,Wicked Choccy's,Interior,Texas,Wicked Choccy's-->Interior,Wicked Choccy's-->Texas
1,Wicked Choccy's,Interior,Illinois,Wicked Choccy's-->Interior,Wicked Choccy's-->Illinois
2,Lot's O' Nuts,Interior,Illinois,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
3,Lot's O' Nuts,Interior,Illinois,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
4,Wicked Choccy's,Atlantic,Pennsylvania,Wicked Choccy's-->Atlantic,Wicked Choccy's-->Pennsylvania


In [31]:
# see the unique routes we have 
print(df['Route_Region'].nunique())
print(df['Route_State'].nunique())

20
196


### • Grouping shipments by Ship Mode

In [32]:
# grouping shipment by ship mode

ship_mode_grouping = df.groupby('Ship Mode').agg({
    "Order ID":"count",
    "Lead Time":"mean",
    "Sales":"sum",
    "Gross Profit":"sum"
}).rename(columns={
    "Order ID":"Total Shipments",
    "Lead Time":"Avg Lead Time"
}).reset_index()

In [33]:
# rounding up to 2 decimal avg lead time and lead time variability
ship_mode_grouping["Avg Lead Time"] = ship_mode_grouping["Avg Lead Time"].round(2)

In [34]:
# see result
ship_mode_grouping

,Ship Mode,Total Shipments,Avg Lead Time,Sales,Gross Profit
0,First Class,1548,1338.28,21319.39,14011.09
1,Same Day,547,1333.44,7113.67,4700.73
2,Second Class,1979,1323.85,27860.22,18306.52
3,Standard Class,6120,1314.33,85490.35,56424.46


In [ ]:
# # saving files
# df.to_csv('nassau_feature_engineering.csv',index=False)
# ship_mode_grouping.to_csv('ship_mode_group.csv',index=False)

# 3. Route Definition & Aggregation

### Loading Dataset

In [35]:
# loading  Dataset in pandas and read it

df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_feature_engineering.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead Time,Factory,Route_Region,Route_State
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Texas
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Illinois
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,Wicked Choccy's,Wicked Choccy's-->Atlantic,Wicked Choccy's-->Pennsylvania


In [36]:
# see info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

###  Each route is defined as: Factory Location → Customer State / Region For each route:

####  • Total shipments
####  • Average shipping lead time
####  • Lead time variability

In [37]:
# route aggregation by factory --> state

route_state_summery = df.groupby('Route_State').agg({
    'Order ID' : 'count',
    'Lead Time' : ['mean','std'],
    'Sales' :'sum',
    'Gross Profit':'sum'
}).reset_index()

# cleaning column names
route_state_summery.columns = [
    'Route_State',
    'Total Shipments',
    'Avg Lead Time',
    'Lead Time Variability',
    'Total Sales',
    'Total Profit'
]

# sorting by avg lead time
route_state_summery = route_state_summery.sort_values(by='Avg Lead Time')

In [38]:
# rounding upto 2 decimal column avg lead time and lead time variabilty
route_state_summery['Avg Lead Time'] = route_state_summery['Avg Lead Time'].round(2)
route_state_summery['Lead Time Variability'] = route_state_summery['Lead Time Variability'].round(2)

In [39]:
#  checking the output
route_state_summery.head()

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit
80,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50
76,Secret Factory-->Nebraska,1,906.0,NaN,2.50,1.30
121,The Other Factory-->Louisiana,1,907.0,NaN,9.75,0.75
115,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75
75,Secret Factory-->Mississippi,1,908.0,NaN,80.00,40.00


In [40]:
# filling NaN values 
route_state_summery['Lead Time Variability'] = route_state_summery['Lead Time Variability'].fillna(0)

In [41]:
#  again checking the output
route_state_summery.head()

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit
80,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50
76,Secret Factory-->Nebraska,1,906.0,0.00,2.50,1.30
121,The Other Factory-->Louisiana,1,907.0,0.00,9.75,0.75
115,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75
75,Secret Factory-->Mississippi,1,908.0,0.00,80.00,40.00


In [42]:
# # saving data in csv file
# route_state_summery.to_csv('nassau_route_states.csv',index=False)

In [43]:
# route aggregation by factory --> region

route_region_summery = df.groupby('Route_Region').agg({
    'Order ID' : 'count',
    'Lead Time' : ['mean','std'],
    'Sales' :'sum',
    'Gross Profit':'sum'
}).round(2).reset_index()

# cleaning column names
route_region_summery.columns = [
    'Route_State',
    'Total Shipments',
    'Avg Lead Time',
    'Lead Time Variability',
    'Total Sales',
    'Total Profit'
]

# sorting by avg lead time
route_region_summery = route_region_summery.sort_values(by='Avg Lead Time')

# filling NaN Values
route_region_summery['Lead Time Variability'] = route_region_summery['Lead Time Variability'].fillna(0)

In [44]:
# lets check it 
route_region_summery.head()

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit
9,Sugar Shack-->Gulf,4,1091.25,210.45,38.39,23.24
14,The Other Factory-->Interior,11,1206.64,274.43,126.75,9.75
13,The Other Factory-->Gulf,19,1272.63,272.73,262.50,26.50
12,The Other Factory-->Atlantic,38,1282.74,262.17,506.75,54.75
6,Secret Factory-->Interior,45,1289.09,311.23,1590.00,806.80


In [45]:
# # saving route_region data in csv file
# route_region_summery.to_csv('nassau_route_region.csv',index=False)

In [ ]:
# # save csv file 
# df.to_csv('nassau_route_defination_&_aggregation.csv',index=False)

# 4. Efficiency Benchmarking

### • Rank routes from fastest to slowest
####  • Identify:
#####   ○ Top 10 most efficient routes
#####  ○ Bottom 10 least efficient routes
#####   • Compare performance across ship modes

###  Loading Dataset

In [46]:
# load dataset and read it
df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_feature_engineering.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead Time,Factory,Route_Region,Route_State
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Texas
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Illinois
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,Wicked Choccy's,Wicked Choccy's-->Atlantic,Wicked Choccy's-->Pennsylvania


In [47]:
# loading routes data and read it
route_state_summery = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_route_states.csv")
route_state_summery.head(10)

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit
0,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50
1,Secret Factory-->Nebraska,1,906.0,0.00,2.50,1.30
2,The Other Factory-->Louisiana,1,907.0,0.00,9.75,0.75
3,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75
4,Secret Factory-->Mississippi,1,908.0,0.00,80.00,40.00
5,Wicked Choccy's-->Maine,2,908.0,0.00,55.25,36.07
6,Secret Factory-->Louisiana,2,908.5,0.71,11.25,5.85
7,Secret Factory-->Delaware,1,909.0,0.00,60.00,30.00
8,Secret Factory-->South Carolina,1,909.0,0.00,6.25,3.25
9,Secret Factory-->Minnesota,1,909.0,0.00,5.00,2.60


### • Rank routes from fastest to slowest

In [48]:
# ranking route from fastet to slowest

# sorting by avg lead time
route_state_summery = route_state_summery.sort_values(by='Avg Lead Time')

# adding rank columns
route_state_summery['Rank'] = range(1,len(route_state_summery) + 1)

# see result
route_state_summery.head(10)

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit,Rank
0,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50,1
1,Secret Factory-->Nebraska,1,906.0,0.00,2.50,1.30,2
2,The Other Factory-->Louisiana,1,907.0,0.00,9.75,0.75,3
3,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75,4
4,Secret Factory-->Mississippi,1,908.0,0.00,80.00,40.00,5
5,Wicked Choccy's-->Maine,2,908.0,0.00,55.25,36.07,6
6,Secret Factory-->Louisiana,2,908.5,0.71,11.25,5.85,7
7,Secret Factory-->Delaware,1,909.0,0.00,60.00,30.00,8
8,Secret Factory-->South Carolina,1,909.0,0.00,6.25,3.25,9
9,Secret Factory-->Minnesota,1,909.0,0.00,5.00,2.60,10


### Top 10 most efficient routes

In [49]:
# top 10 routes most efficient routes

top_10_routes = route_state_summery.head(10)
top_10_routes

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit,Rank
0,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50,1
1,Secret Factory-->Nebraska,1,906.0,0.00,2.50,1.30,2
2,The Other Factory-->Louisiana,1,907.0,0.00,9.75,0.75,3
3,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75,4
4,Secret Factory-->Mississippi,1,908.0,0.00,80.00,40.00,5
5,Wicked Choccy's-->Maine,2,908.0,0.00,55.25,36.07,6
6,Secret Factory-->Louisiana,2,908.5,0.71,11.25,5.85,7
7,Secret Factory-->Delaware,1,909.0,0.00,60.00,30.00,8
8,Secret Factory-->South Carolina,1,909.0,0.00,6.25,3.25,9
9,Secret Factory-->Minnesota,1,909.0,0.00,5.00,2.60,10


###  Bottom 10 least efficient routes

In [50]:
# bottom 10 least efficient routes

bottom_10_routes = route_state_summery.tail(10).sort_values(by='Avg Lead Time',ascending=False)

# fixing  rank for bottom 10 routes
bottom_10_routes['Rank'] = range(len(route_state_summery),len(route_state_summery)-10,-1)

bottom_10_routes

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit,Rank
195,Sugar Shack-->New Jersey,1,1642.0,0.00,5.97,3.72,196
193,Secret Factory-->New Hampshire,1,1641.0,0.00,6.25,3.25,195
194,Sugar Shack-->Connecticut,1,1641.0,0.00,3.98,2.48,194
192,Wicked Choccy's-->West Virginia,2,1639.0,0.00,25.25,16.47,193
191,Lot's O' Nuts-->North Dakota,5,1638.2,1.64,57.16,39.66,192
187,The Other Factory-->Nevada,1,1638.0,0.00,16.25,1.25,191
188,Sugar Shack-->California,1,1638.0,0.00,6.00,2.80,190
190,Secret Factory-->Connecticut,1,1638.0,0.00,2.50,1.30,189
189,Sugar Shack-->Washington,1,1638.0,0.00,1.99,1.24,188
186,Secret Factory-->Kentucky,2,1637.5,0.71,46.25,23.25,187


### Compare performance across ship modes

In [51]:
# see columns
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time', 'Factory',
       'Route_Region', 'Route_State'],
      dtype='object')

In [52]:
# camparigion of performance acrross ship mode

ship_mode_prf = df.groupby("Ship Mode").agg({
    "Order ID":"count",
    "Lead Time" : ["mean","std"],
    "Sales":"sum",
    "Gross Profit":"sum"
}).round(2).reset_index()

ship_mode_prf.columns = [
    "Ship Mode",
    "Total Shipments",
    "Avg Lead Time",
    "Lead Time Variability",
    "Total Sales",
    "Total Profit"
]

# sorting values by avg lead time
ship_mode_prf = ship_mode_prf.sort_values(by='Avg Lead Time')

In [53]:
# lets see ship_mode_prf
ship_mode_prf

,Ship Mode,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit
3,Standard Class,6120,1314.33,262.40,85490.35,56424.46
2,Second Class,1979,1323.85,261.81,27860.22,18306.52
1,Same Day,547,1333.44,253.81,7113.67,4700.73
0,First Class,1548,1338.28,265.63,21319.39,14011.09


In [54]:
# which ship mode fastest
fastest_mode = ship_mode_prf.iloc[0]['Ship Mode']
slowest_mode = ship_mode_prf.iloc[-1]['Ship Mode']

# lets see
print('fastest_mode:',fastest_mode)
print('slowest_mode:',slowest_mode)

fastest_mode: Standard Class
slowest_mode: First Class


In [55]:
# calculating route efficiency score

# higher lead time = lower score
# flipping it so best route get score 1

min_lt = route_state_summery['Avg Lead Time'].min()
max_lt = route_state_summery['Avg Lead Time'].max()


# score = 1-(lead Time - min)/(max-min)

route_state_summery['Efficiency Score'] = 1 - (route_state_summery['Avg Lead Time'] - min_lt)/(max_lt-min_lt)

# rounding of upto 2 decimal
route_state_summery['Efficiency Score'] = route_state_summery['Efficiency Score'].round(2)

# see result
# fast route = high score becaouse we flip it before
route_state_summery.head(10)

,Route_State,Total Shipments,Avg Lead Time,Lead Time Variability,Total Sales,Total Profit,Rank,Efficiency Score
0,Secret Factory-->New Mexico,2,906.0,2.83,12.50,6.50,1,1.0
1,Secret Factory-->Nebraska,1,906.0,0.00,2.50,1.30,2,1.0
2,The Other Factory-->Louisiana,1,907.0,0.00,9.75,0.75,3,1.0
3,The Other Factory-->Connecticut,2,907.5,2.12,9.75,0.75,4,1.0
4,Secret Factory-->Mississippi,1,908.0,0.00,80.00,40.00,5,1.0
5,Wicked Choccy's-->Maine,2,908.0,0.00,55.25,36.07,6,1.0
6,Secret Factory-->Louisiana,2,908.5,0.71,11.25,5.85,7,1.0
7,Secret Factory-->Delaware,1,909.0,0.00,60.00,30.00,8,1.0
8,Secret Factory-->South Carolina,1,909.0,0.00,6.25,3.25,9,1.0
9,Secret Factory-->Minnesota,1,909.0,0.00,5.00,2.60,10,1.0


In [56]:
# calculating is delay or not

# average lead time of all shipments
thresholds = df['Lead Time'].mean().round(1)
print(thresholds)

# marking shipments delays or not
# if lead time is more than average = true(delayed)
# if Lead time is below average = false(not delayed)
df['Is Delay'] = df['Lead Time'] > thresholds

# see result
df[['Route_State','Lead Time','Is Delay']].head(10)

1320.8


,Route_State,Lead Time,Is Delay
0,Wicked Choccy's-->Texas,909,False
1,Wicked Choccy's-->Illinois,909,False
2,Lot's O' Nuts-->Illinois,909,False
3,Lot's O' Nuts-->Illinois,909,False
4,Wicked Choccy's-->Pennsylvania,912,False
5,Lot's O' Nuts-->Kentucky,909,False
6,Wicked Choccy's-->Kentucky,909,False
7,Wicked Choccy's-->Georgia,906,False
8,Lot's O' Nuts-->Kentucky,909,False
9,Wicked Choccy's-->Kentucky,909,False


In [57]:
# counting total and delayed shipments per route
# total shipments per route
total_shipments = df.groupby('Route_State')['Is Delay'].count()
total_shipments.head(10)


Route_State
Lot's O' Nuts-->Alabama                   34
Lot's O' Nuts-->Alberta                   16
Lot's O' Nuts-->Arizona                  111
Lot's O' Nuts-->Arkansas                  31
Lot's O' Nuts-->British Columbia          18
Lot's O' Nuts-->California              1125
Lot's O' Nuts-->Colorado                 103
Lot's O' Nuts-->Connecticut               47
Lot's O' Nuts-->Delaware                  43
Lot's O' Nuts-->District Of Columbia       8
Name: Is Delay, dtype: int64

In [58]:
# delayed shipments per route
delayed_shipments = df.groupby('Route_State')['Is Delay'].sum()
delayed_shipments.head(10)

Route_State
Lot's O' Nuts-->Alabama                   9
Lot's O' Nuts-->Alberta                   4
Lot's O' Nuts-->Arizona                  32
Lot's O' Nuts-->Arkansas                  9
Lot's O' Nuts-->British Columbia          6
Lot's O' Nuts-->California              372
Lot's O' Nuts-->Colorado                 37
Lot's O' Nuts-->Connecticut              24
Lot's O' Nuts-->Delaware                 10
Lot's O' Nuts-->District Of Columbia      2
Name: Is Delay, dtype: int64

In [59]:
# calculating delay percentage

delay_rate = pd.DataFrame({
    'Total': total_shipments,
    'Delayed':delayed_shipments
})

# delay % = (delayed_shipments/total_oshipments)*100

delay_rate['Delay %'] = ((delay_rate['Delayed']/delay_rate['Total'])*100).round(2)
# reseting index because routes become normal column
delay_rate = delay_rate.reset_index()

# see result
delay_rate.head(10)

,Route_State,Total,Delayed,Delay %
0,Lot's O' Nuts-->Alabama,34,9,26.47
1,Lot's O' Nuts-->Alberta,16,4,25.00
2,Lot's O' Nuts-->Arizona,111,32,28.83
3,Lot's O' Nuts-->Arkansas,31,9,29.03
4,Lot's O' Nuts-->British Columbia,18,6,33.33
5,Lot's O' Nuts-->California,1125,372,33.07
6,Lot's O' Nuts-->Colorado,103,37,35.92
7,Lot's O' Nuts-->Connecticut,47,24,51.06
8,Lot's O' Nuts-->Delaware,43,10,23.26
9,Lot's O' Nuts-->District Of Columbia,8,2,25.00


In [60]:
# see top 10 worst routes

delay_rate = delay_rate.sort_values('Delay %',ascending=False)
delay_rate.head()

,Route_State,Total,Delayed,Delay %
63,Secret Factory-->Connecticut,1,1,100.0
37,Lot's O' Nuts-->North Dakota,5,5,100.0
54,Lot's O' Nuts-->West Virginia,2,2,100.0
69,Secret Factory-->Kentucky,2,2,100.0
103,Sugar Shack-->New Jersey,1,1,100.0


In [61]:
# # saving csv file
# df.to_csv('nassau_efficiency_benchmarking.csv',index=False)

# 5. Geographic Bottleneck Analysis

###  • Identify regions with:
#####  ○ High average lead time
#####  ○ High shipment volume + poor performance
##### • Detect congestion-prone states or regions

## Loading Dataset

In [62]:
#  Loading Dataset and read it

df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_efficiency_benchmarking.csv")
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product Name,Sales,Units,Gross Profit,Cost,Lead Time,Factory,Route_Region,Route_State,Is Delay
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Texas,False
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Illinois,False
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois,False
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois,False
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,Wicked Choccy's,Wicked Choccy's-->Atlantic,Wicked Choccy's-->Pennsylvania,False


In [63]:
# see columns
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time', 'Factory',
       'Route_Region', 'Route_State', 'Is Delay'],
      dtype='object')

In [64]:
# lets first see which regions are slowest (grouibg by region)
#  region wise analysis

region_analysis = df.groupby('Region').agg({
    'Order ID' : 'count',
    'Lead Time' : ['mean','std']
}).round(2).reset_index()

region_analysis.columns = [
    'Region',
    'Total Shipments',
    'Avg Lead Time',
    'Lead Time Variability'
]

region_analysis = region_analysis.sort_values(by='Avg Lead Time',ascending=False)

In [65]:
# see the result

region_analysis

,Region,Total Shipments,Avg Lead Time,Lead Time Variability
2,Interior,2335,1323.09,262.69
0,Atlantic,2986,1322.75,256.54
3,Pacific,3253,1322.19,266.47
1,Gulf,1620,1311.37,264.73


In [66]:
# identifing  botleneck regions

botleneck_regions = region_analysis[
(region_analysis['Avg Lead Time'] > region_analysis['Avg Lead Time'].mean()) &
(region_analysis['Total Shipments'] > region_analysis['Total Shipments'].mean())
]

# printing it

botleneck_regions

,Region,Total Shipments,Avg Lead Time,Lead Time Variability
0,Atlantic,2986,1322.75,256.54
3,Pacific,3253,1322.19,266.47


In [67]:
# state wise analysis

state_analysis = df.groupby('State/Province').agg({
    'Order ID' : 'count',
    'Lead Time' : ['mean','std']
}).round(2).reset_index()

state_analysis.columns = [
    'State',
    'Total Shipments',
    'Avg Lead Time',
    'Lead Time Variability'
]

# sorting slowest route first
state_analysis = state_analysis.sort_values(by='Avg Lead Time',ascending=False)

In [68]:
# printing  top 10 slowest routes by state
state_analysis.head(10)

,State,Total Shipments,Avg Lead Time,Lead Time Variability
56,West Virginia,4,1638.00,2.00
37,North Dakota,7,1637.86,1.46
47,Saskatchewan,2,1457.00,258.80
20,Manitoba,12,1455.33,191.14
15,Iowa,30,1443.90,229.80
33,New Mexico,37,1441.84,267.20
53,Vermont,11,1438.91,191.00
44,Prince Edward Island,10,1420.30,255.81
49,South Dakota,12,1395.92,360.35
50,Tennessee,183,1391.49,248.20


In [69]:
# defining   high volume = above average number of shipments
avg_shipments = state_analysis['Total Shipments'].mean().round(2)
print('Average shipmetns per state:',avg_shipments)

# definig poor performance = above average lead time
avg_lt = state_analysis['Avg Lead Time'].mean().round(2)
print('Avrarage lead time',avg_lt)

Average shipmetns per state: 172.78
Avrarage lead time 1328.58


In [70]:
# filtering high volume and slow delivery = true bottlenecks
bottlenecks = state_analysis[
(state_analysis['Total Shipments'] > avg_shipments) & 
(state_analysis['Avg Lead Time'] > avg_lt)
]

# sort by worst lead time first
bottlenecks =  bottlenecks.sort_values('Avg Lead Time',ascending=False)

# bottlenecks by high volume and low performance
print('bottlenecks states according ("high volume and low performance")')
# see result

bottlenecks.head()

bottlenecks states according ("high volume and low performance")


,State,Total Shipments,Avg Lead Time,Lead Time Variability
50,Tennessee,183,1391.49,248.20
55,Washington,506,1360.66,272.12
11,Georgia,184,1338.64,249.18
6,Colorado,182,1337.19,249.68
36,North Carolina,249,1334.88,254.99


In [71]:
# filtering high volume and slow delivery = true bottlenecks
bottlenecks = state_analysis[
(state_analysis['Total Shipments'] > avg_shipments) & 
(state_analysis['Avg Lead Time'] > avg_lt)
]

# sort by worst lead time first
bottlenecks =  bottlenecks.sort_values('Avg Lead Time',ascending=False)

# bottlenecks by high volume and low performance
print('bottlenecks states according ("high volume and low performance")')
# see result

bottlenecks.head()

bottlenecks states according ("high volume and low performance")


,State,Total Shipments,Avg Lead Time,Lead Time Variability
50,Tennessee,183,1391.49,248.20
55,Washington,506,1360.66,272.12
11,Georgia,184,1338.64,249.18
6,Colorado,182,1337.19,249.68
36,North Carolina,249,1334.88,254.99


In [72]:
# labeling states into 4 category based on volume and lead time

def label_state(row):
    high_volume = row['Total Shipments'] > avg_shipments
    slow        = row['Avg Lead Time'] > avg_lt


    if high_volume and slow:                     # worst performance
        return "high volume + slow"
    elif high_volume and not slow:               # great  performance
        return "high volume + fast"
    elif not high_volume and slow:               # minir issues        
        return "low voulme + slow"
    else:
        return "low volume + fast"               # fine

# applying function
state_analysis['Category'] = state_analysis.apply(label_state,axis=1)

# see result

state_analysis.head(10)

,State,Total Shipments,Avg Lead Time,Lead Time Variability,Category
56,West Virginia,4,1638.00,2.00,low voulme + slow
37,North Dakota,7,1637.86,1.46,low voulme + slow
47,Saskatchewan,2,1457.00,258.80,low voulme + slow
20,Manitoba,12,1455.33,191.14,low voulme + slow
15,Iowa,30,1443.90,229.80,low voulme + slow
33,New Mexico,37,1441.84,267.20,low voulme + slow
53,Vermont,11,1438.91,191.00,low voulme + slow
44,Prince Edward Island,10,1420.30,255.81,low voulme + slow
49,South Dakota,12,1395.92,360.35,low voulme + slow
50,Tennessee,183,1391.49,248.20,high volume + slow


In [73]:
# detecting congestion prone states 
#by three parameter Avg lead time ,lead time variabilty and Total shipments

congestion_states = state_analysis[
(state_analysis['Avg Lead Time'] > state_analysis['Avg Lead Time'].mean()) &
(state_analysis['Lead Time Variability'] > state_analysis['Lead Time Variability'].mean()) &
(state_analysis['Total Shipments'] > state_analysis['Total Shipments'].mean())
]

# printing it

congestion_states

,State,Total Shipments,Avg Lead Time,Lead Time Variability,Category
55,Washington,506,1360.66,272.12,high volume + slow
36,North Carolina,249,1334.88,254.99,high volume + slow


In [74]:
# # saving csv file 
# df.to_csv('nassau_bottleneck_analysis.csv',index=False)

# 6. Ship Mode Performance Analysis

###  • Compare shipping efficiency by:
#### ○ Standard shipping
#### ○ Expedited shipping
#### • Evaluate cost-time tradeoffs (descriptive)

### Laoding Dataset

In [75]:
# load dataset and read it

df = pd.read_csv(r"C:\Users\91935\jupyter\projects\route_efficeincey\nassau_bottleneck_analysis.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product Name,Sales,Units,Gross Profit,Cost,Lead Time,Factory,Route_Region,Route_State,Is Delay
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Texas,False
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909,Wicked Choccy's,Wicked Choccy's-->Interior,Wicked Choccy's-->Illinois,False
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois,False
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909,Lot's O' Nuts,Lot's O' Nuts-->Interior,Lot's O' Nuts-->Illinois,False
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912,Wicked Choccy's,Wicked Choccy's-->Atlantic,Wicked Choccy's-->Pennsylvania,False


In [76]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time', 'Factory',
       'Route_Region', 'Route_State', 'Is Delay'],
      dtype='object')

In [77]:
# catagorizing ship mode

def categorize_shipping(mode):
    if 'Standard' in mode:
        return 'Standard Shipping'
    else:
        return 'Expedited Shipping'

df['Shipping Category'] = df['Ship Mode'].apply(categorize_shipping)

# camparing performance

efficiency_comparison = df.groupby('Shipping Category').agg({
    'Order ID':'count',
    'Lead Time': ['mean','std'],
    'Sales': 'sum',
    'Cost' : 'mean',
    'Gross Profit':'mean'
}).round(2).reset_index()

efficiency_comparison.columns = [
    'Shipping Category',
    'Total Shipments',
    'Avg Lead Time',
    'Lead Time Variabilty',
    'Total Sales',
    'Avg Cost',
    'Avg Profit'
]


# evaluating cost - time tradeoffs

efficiency_comparison['Avg Sales Per Shipments'] = (
    efficiency_comparison['Total Sales']/efficiency_comparison['Total Shipments']
).round(2)


In [78]:
# printing efficiency_camparison

efficiency_comparison

,Shipping Category,Total Shipments,Avg Lead Time,Lead Time Variabilty,Total Sales,Avg Cost,Avg Profit,Avg Sales Per Shipments
0,Expedited Shipping,4074,1330.62,262.24,56293.28,4.73,9.09,13.82
1,Standard Shipping,6120,1314.33,262.40,85490.35,4.75,9.22,13.97


In [ ]:
# # save csv file

# df.to_csv('ship_mode_performance_nanalysis.csv',index=False)